In [3]:
import nltk
import pandas as pd
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline



nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

data = pd.read_csv(
    "Dataset/SMSSpamCollection",
    sep="\t",
    header=None,
    names=["label", "message"]
)

data["message"] = data["message"].str.lower()

data["message"] = data["message"].apply(lambda x: re.sub(r"[^a-z\s$!]", "", x))

data["message"] = data["message"].apply(word_tokenize)

stop_words = set(stopwords.words("english"))

data["message"] = data["message"].apply(lambda x: [word for word in x if word not in stop_words])


# Stem each token to reduce words to their base form
stemmer = PorterStemmer()
data["message"] = data["message"].apply(lambda x: [stemmer.stem(word) for word in x])

# Rejoin tokens into a single string for feature extraction
data["message"] = data["message"].apply(lambda x: " ".join(x))

y = data["label"].apply(lambda x: 1 if x == "spam" else 0)  # Converting labels to 1 and 0


vectorizer = CountVectorizer(min_df=1, max_df=0.9, ngram_range=(1, 2))

# Build the pipeline by combining vectorization and classification
pipeline = Pipeline([
    ("vectorizer", vectorizer),
    ("classifier", MultinomialNB())
])


# Define the parameter grid for hyperparameter tuning
param_grid = {
    "classifier__alpha": [0.01, 0.1, 0.15, 0.2, 0.25, 0.5, 0.75, 1.0]
}

# Perform the grid search with 5-fold cross-validation and the F1-score as metric
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="f1"
)

# Fit the grid search on the full dataset
grid_search.fit(data["message"], y)

# Extract the best model identified by the grid search
best_model = grid_search.best_estimator_
print("Best model parameters:", grid_search.best_params_)



[nltk_data] Downloading package punkt to /home/noob/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/noob/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/noob/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Best model parameters: {'classifier__alpha': 0.2}
